# Modelo de datos y consultas SQL

Las tablas de los notebooks anteriores ya tienen los datos limpios. Falta convertirlos en un modelo que se pueda consultar: una base de datos con las tablas de hechos y un conjunto de vistas SQL que responden las preguntas habituales (casos por año, por provincia, por grupo de edad, incidencia, variación) y que además sirven de fuente al dashboard.

Este notebook hace cinco cosas: une los casos con el clima de cada provincia y semana, arma una tabla analítica con una fila por provincia y semana (incluidas las semanas sin casos), carga todo en una base SQLite, crea las vistas y las exporta para Power BI. Verifica cada vista contra el mismo cálculo hecho en pandas, porque una consulta SQL que corre sin error no garantiza que dé el número correcto.

Las preguntas que ordenan el recorrido:

1. ¿Cómo se une el dengue, que se notifica por semana epidemiológica, con un clima que viene por mes?
2. ¿Cómo se representan las semanas sin casos, que en la fuente simplemente no aparecen?
3. ¿Las vistas devuelven lo mismo que un cálculo independiente?

**Entradas:** `data/interim/dengue_casos_limpio.csv` (notebook 01), y las tablas de clima en `data/processed/` (notebook 02).
**Salidas:** `data/processed/panel_provincia_semana.csv`, `sql/dengue.db`, `sql/vistas_dengue.sql` y los CSV de `powerbi/`, que carga el dashboard.

## 1. Preparación

### 1.1 Librerías y rutas

In [1]:
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def encontrar_raiz(archivo=("data", "raw", "Dengue y Zika - Dic. 2019.xlsx")):
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        for candidata in (base, base / "vigilancia-dengue-argentina"):
            if candidata.joinpath(*archivo).exists():
                return candidata
    raise FileNotFoundError("No se encontró data/raw/" + archivo[-1] + " desde " + str(cwd))

ROOT = encontrar_raiz()
RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
IMAGES = ROOT / "images"
for carpeta in (INTERIM, PROCESSED, IMAGES):
    carpeta.mkdir(parents=True, exist_ok=True)

# Paleta del dashboard de Power BI (tema "Dengue Portfolio")
AZUL_OSCURO, ROJO, AZUL, NARANJA, VERDE, VIOLETA, AMARILLO, GRIS = (
    "#2C3E50", "#B71C1C", "#1F6FB2", "#E08E45", "#6B8E23", "#8E44AD", "#F1C40F", "#7F8C8D")
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#9CA3AF", "axes.labelcolor": "#1F2937", "text.color": "#1F2937",
    "xtick.color": "#374151", "ytick.color": "#374151",
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.prop_cycle": plt.cycler(color=[AZUL_OSCURO, ROJO, AZUL, NARANJA, VERDE, VIOLETA, AMARILLO, GRIS]),
    "axes.grid": True, "grid.color": "#E5E7EB", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "font.size": 10, "legend.frameon": False,
})

### 1.2 Carga

In [2]:
casos = pd.read_csv(INTERIM / "dengue_casos_limpio.csv", parse_dates=["SE_inicio", "SE_fin"])
clima_mes = pd.read_csv(PROCESSED / "clima_provincia_mensual.csv")
clima_sem = pd.read_csv(PROCESSED / "clima_provincia_semanal.csv")
print(f"Casos limpios: {casos.shape} | clima mensual: {clima_mes.shape} | clima semanal: {clima_sem.shape}")

Casos limpios: (70194, 13) | clima mensual: (2208, 8) | clima semanal: (9591, 8)


## 2. Tabla de hechos: casos con clima

### 2.1 A qué mes pertenece una semana epidemiológica

Los casos vienen por semana y el clima mensual por mes, así que hay que decidir a qué mes pertenece cada semana. Hay semanas que cruzan dos meses. La regla es la misma que usa el calendario ISO: **la semana pertenece al mes de su jueves**, o sea, al mes que concentra la mayoría de sus días. Una semana que empieza un lunes 29 y termina un domingo 4 queda en el mes donde caen cuatro de sus siete días.

In [3]:
# Cada semana epidemiológica se asigna al mes de su jueves (regla ISO): así una semana que cruza dos meses
# cae en el que concentra la mayoría de sus días.
referencia = casos["SE_inicio"] + pd.Timedelta(days=3)
casos["año_clima"] = referencia.dt.year
casos["mes"] = referencia.dt.month

casos[["año", "SE", "SE_inicio", "año_clima", "mes"]].drop_duplicates(["año", "SE"]).sort_values(["año", "SE"]).head(8)

,año,SE,SE_inicio,año_clima,mes
0,2018,1,2018-01-01,2018,1
1,2018,2,2018-01-08,2018,1
2,2018,3,2018-01-15,2018,1
3,2018,4,2018-01-22,2018,1
4,2018,5,2018-01-29,2018,2
6,2018,6,2018-02-05,2018,2
8,2018,7,2018-02-12,2018,2
12,2018,8,2018-02-19,2018,2


### 2.2 Unión con el clima mensual

La unión se hace por provincia, año y mes. Se usa `how="left"`: los casos son la tabla principal y nunca se pierden filas por falta de clima. El `assert` comprueba que la unión no multiplicó ni perdió registros, que es el error silencioso más común en este tipo de cruces.

In [4]:
clima_unido = clima_mes.rename(columns={"año": "año_clima", "precip_mm": "precip_media", "dias_lluvia": "dias_pp_mas_1mm"})
clima_unido = clima_unido.drop(columns="n_estaciones")

fact = casos.merge(clima_unido, on=["provincia", "año_clima", "mes"], how="left")
assert len(fact) == len(casos), "La unión con clima duplicó o perdió filas"

con_prov = fact["provincia"].notna()
print("Filas con provincia:", int(con_prov.sum()), "| sin clima entre ellas:", int(fact.loc[con_prov, "temp_media"].isna().sum()))
print("Filas sin provincia (no tienen clima asignado):", int((~con_prov).sum()))

Filas con provincia: 69624 | sin clima entre ellas: 0
Filas sin provincia (no tienen clima asignado): 570


Las 570 filas sin provincia corresponden a casos que la fuente no asigna a ninguna (28.370 casos, el 3,5 % del total). Cuentan en los totales nacionales, pero no tienen clima y no entran en ningún cálculo provincial. Todas las filas con provincia quedaron con clima.

### 2.3 Tabla final

In [5]:
COLUMNAS_FACT = ["id_dpto", "departamento", "id_prov", "provincia", "cant_habitantes", "año", "SE", "SE_inicio", "SE_fin",
                 "temp_media", "precip_media", "hum_rel", "dias_pp_mas_1mm", "evento", "id_grupo_edad", "grupo_edad",
                 "cant_casos", "mes"]
fact = fact[COLUMNAS_FACT].copy()
for col in ["temp_media", "precip_media", "hum_rel", "dias_pp_mas_1mm"]:
    fact[col] = fact[col].round(2)
fact["cant_habitantes"] = fact["cant_habitantes"].astype("Int64")

print(f"Tabla de hechos: {fact.shape[0]:,} filas, {fact.shape[1]} columnas".replace(",", "."))
fact.head(3)

Tabla de hechos: 70.194 filas. 18 columnas


,id_dpto,departamento,id_prov,provincia,cant_habitantes,año,SE,SE_inicio,SE_fin,temp_media,precip_media,hum_rel,dias_pp_mas_1mm,evento,id_grupo_edad,grupo_edad,cant_casos,mes
0,270,formosa,34.0,Formosa,606041,2018,1,2018-01-01,2018-01-07,27.74,214.55,73.39,12.0,Dengue,8,De 20 a 24 años,3,1
1,355,capital,54.0,Misiones,1280960,2018,2,2018-01-08,2018-01-14,24.73,343.25,79.73,14.5,Dengue,9,De 25 a 34 años,1,1
2,270,formosa,34.0,Formosa,606041,2018,3,2018-01-15,2018-01-21,27.74,214.55,73.39,12.0,Dengue,11,De 45 a 64 años,1,1


La tabla de hechos tiene 70.194 filas, una por departamento, semana y grupo de edad, y reúne los 814.349 casos del período.

## 3. Panel provincia-semana

La fuente solo registra las semanas en que un departamento notificó casos. Una semana sin casos no aparece, y eso crea un problema para cualquier análisis de series de tiempo: promediar solo las semanas con casos sobrestima la incidencia, y un rezago entre clima y casos no se puede calcular si faltan semanas.

La solución es armar un **panel completo**: una fila para cada una de las 23 provincias y para cada semana observada, con cero casos donde no hubo notificación. Se consideran solo las semanas que cada año realmente cubre (hasta la última con registros), porque en un año parcial como 2025 las semanas posteriores no son ceros: son períodos que todavía no se informaron.

In [6]:
# Tabla analítica: una fila por provincia y semana, con ceros explícitos donde no hubo casos notificados.
# Solo se incluyen las semanas que cada año realmente cubre (hasta la última SE con registros).
semanas_obs = casos.groupby("año")["SE"].max()
grilla = pd.DataFrame(
    [(a, s) for a, smax in semanas_obs.items() for s in range(1, smax + 1)], columns=["año", "SE"]
)
provincias = pd.read_csv(RAW / "poblacion_provincias_censo2022.csv").rename(columns={"habitantes": "cant_habitantes"})
panel = grilla.merge(provincias, how="cross")

semanal_casos = (casos.dropna(subset=["provincia"])
                 .groupby(["provincia", "año", "SE"], as_index=False)["cant_casos"].sum())
panel = panel.merge(semanal_casos, on=["provincia", "año", "SE"], how="left")
panel["cant_casos"] = panel["cant_casos"].fillna(0).astype(int)

inicio = pd.to_datetime(panel["año"].astype(str) + "-W" + panel["SE"].astype(str).str.zfill(2) + "-1", format="%G-W%V-%u")
panel.insert(3, "SE_inicio", inicio)
panel["incidencia_100k"] = panel["cant_casos"] / panel["cant_habitantes"] * 100_000

panel = panel.merge(clima_sem.rename(columns={"precip_mm": "precip_sem", "dias_lluvia": "dias_lluvia_sem"})
                    .drop(columns="n_estaciones"), on=["provincia", "año", "SE"], how="left")
panel = panel.sort_values(["provincia", "año", "SE"]).reset_index(drop=True)

print(f"Panel: {panel.shape[0]:,} filas = 23 provincias x {len(grilla)} semanas observadas")
print("Semanas sin clima:", int(panel["temp_media"].isna().sum()))
print("Casos en el panel:", int(panel["cant_casos"].sum()), "| casos con provincia en la tabla de hechos:",
      int(fact.loc[fact["provincia"].notna(), "cant_casos"].sum()))
assert panel["cant_casos"].sum() == fact.loc[fact["provincia"].notna(), "cant_casos"].sum()
panel.to_csv(PROCESSED / "panel_provincia_semana.csv", index=False, encoding="utf-8")
panel.head()

Panel: 8,234 filas = 23 provincias x 358 semanas observadas
Semanas sin clima: 0
Casos en el panel: 785979 | casos con provincia en la tabla de hechos: 785979


,año,SE,provincia,SE_inicio,id_prov,cant_habitantes,cant_casos,incidencia_100k,temp_media,precip_sem,hum_rel,dias_lluvia_sem
0,2018,1,Buenos Aires,2018-01-01,6,20689665,0,0.0,23.637662,15.031818,48.207792,1.000000
1,2018,2,Buenos Aires,2018-01-08,6,20689665,0,0.0,25.556494,32.954545,59.883117,1.500000
2,2018,3,Buenos Aires,2018-01-15,6,20689665,0,0.0,24.375325,4.654545,61.889610,0.636364
3,2018,4,Buenos Aires,2018-01-22,6,20689665,0,0.0,22.872078,10.763636,61.655844,0.772727
4,2018,5,Buenos Aires,2018-01-29,6,20689665,0,0.0,25.115584,5.559091,59.019481,0.409091


El panel tiene 8.234 filas (23 provincias × 358 semanas) y todas cuentan con temperatura, humedad y lluvia semanales. Suma 785.979 casos, exactamente los de la tabla de hechos con provincia: el `assert` lo verifica, y la diferencia con el total (28.370 casos) son los que no tienen provincia.

Este panel es la tabla que usan el análisis de clima y brotes (notebook 04) y el pronóstico (notebook 05).

## 4. Base de datos SQLite

### 4.1 Definición de las tablas

Se definen las tablas con sus tipos explícitos en lugar de dejar que pandas los infiera. Los identificadores (`id_dpto`, `id_prov`, `SE`, `año`) se guardan como texto: son códigos, no cantidades, y mantener los tipos de la versión anterior de la base evita romper las relaciones del modelo del dashboard.

In [7]:
DB = ROOT / "sql" / "dengue.db"
DB.parent.mkdir(exist_ok=True)

DDL_VIG = """
CREATE TABLE vig_dengue (
    id_dpto TEXT, departamento TEXT, id_prov TEXT, provincia TEXT, cant_habitantes INTEGER,
    "año" TEXT, SE TEXT, SE_inicio TIMESTAMP, SE_fin TIMESTAMP,
    temp_media REAL, precip_media REAL, hum_rel REAL, "dias_pp_mas_1mm" REAL,
    evento TEXT, id_grupo_edad TEXT, grupo_edad TEXT, cant_casos INTEGER, mes INTEGER
)
"""
DDL_PANEL = """
CREATE TABLE panel_provincia_semana (
    "año" INTEGER, SE INTEGER, provincia TEXT, SE_inicio TIMESTAMP, id_prov INTEGER, cant_habitantes INTEGER,
    cant_casos INTEGER, incidencia_100k REAL, temp_media REAL, precip_sem REAL, hum_rel REAL, dias_lluvia_sem REAL
)
"""

### 4.2 Carga

Antes de cargar se elimina todo lo existente, así el notebook puede ejecutarse las veces que haga falta sin acumular datos duplicados.

In [8]:
con = sqlite3.connect(DB)
# Se reconstruye todo desde cero: el notebook puede correrse las veces que haga falta
for tipo, nombre in con.execute("SELECT type, name FROM sqlite_master WHERE type IN ('view', 'table')").fetchall():
    con.execute(f'DROP {tipo} IF EXISTS "{nombre}"')

con.execute(DDL_VIG)
con.execute(DDL_PANEL)

bd = fact.copy()
for col in ["SE_inicio", "SE_fin"]:
    bd[col] = bd[col].dt.strftime("%Y-%m-%d %H:%M:%S")
# Tipos TEXT heredados de la versión anterior de la base (el modelo de Power BI los usa tal cual)
for col in ["id_dpto", "id_prov", "año", "SE", "id_grupo_edad"]:
    texto = bd[col].astype("Int64").astype("string")
    bd[col] = texto.astype(object).where(texto.notna(), None)
bd["grupo_edad"] = bd["grupo_edad"].astype(str)
bd.to_sql("vig_dengue", con, if_exists="append", index=False)

pn = panel[["año", "SE", "provincia", "SE_inicio", "id_prov", "cant_habitantes", "cant_casos", "incidencia_100k",
            "temp_media", "precip_sem", "hum_rel", "dias_lluvia_sem"]].copy()
pn["SE_inicio"] = pn["SE_inicio"].dt.strftime("%Y-%m-%d %H:%M:%S")
pn.to_sql("panel_provincia_semana", con, if_exists="append", index=False)
con.commit()

for t in ("vig_dengue", "panel_provincia_semana"):
    print(t, con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0], "filas")

vig_dengue 70194 filas
panel_provincia_semana 8234 filas


## 5. Vistas SQL

Las 13 vistas condensan las consultas que alimentan el dashboard y el análisis. Conservan los nombres y las columnas de la versión anterior del proyecto, para que cualquier informe que las lea siga funcionando; lo que cambió es el contenido, que ahora sale de datos corregidos y de clima real por semana.

Hay decisiones de diseño que vale la pena explicitar:

- **Incidencia por provincia:** casos acumulados del período por cada 100.000 habitantes. Se usa `NULLIF` en el divisor para que una población en cero devuelva nulo y no un error.
- **Departamentos:** el nombre no es una clave, porque "capital" existe en 11 provincias. Por eso el ranking agrupa por provincia y departamento y los muestra juntos.
- **Variación semanal y anual:** usan la función de ventana `LAG`. La semanal parte del panel completo, para que la semana previa sea realmente la anterior y no la anterior *con casos*.
- **Clima y casos:** las vistas climáticas salen del panel, que ya trae el clima semanal real de la provincia.

In [9]:
VISTAS = {}

VISTAS["vista_casos_por_anio"] = """
SELECT "año", SUM(cant_casos) AS casos_acumulados
FROM vig_dengue
GROUP BY "año"
ORDER BY "año"
"""

VISTAS["vista_casos_por_provincia"] = """
SELECT provincia, SUM(cant_casos) AS casos_acumulados
FROM vig_dengue
WHERE provincia IS NOT NULL
GROUP BY provincia
ORDER BY casos_acumulados DESC
"""

VISTAS["vista_casos_por_grupo_edad"] = """
SELECT grupo_edad, SUM(cant_casos) AS casos_acumulados
FROM vig_dengue
GROUP BY grupo_edad
ORDER BY casos_acumulados
"""

VISTAS["vista_proporcion_casos_grupo_edad"] = """
SELECT grupo_edad,
       SUM(cant_casos) AS casos_acumulados,
       SUM(cant_casos) * 100.0 / (SELECT SUM(cant_casos) FROM vig_dengue) AS proporcion
FROM vig_dengue
GROUP BY grupo_edad
ORDER BY casos_acumulados DESC
"""

VISTAS["vista_top_departamentos"] = """
-- El nombre del departamento se repite entre provincias ("capital" existe en 11), por eso se muestra junto a la provincia
SELECT departamento || ' (' || provincia || ')' AS departamento, SUM(cant_casos) AS casos_acumulados
FROM vig_dengue
WHERE departamento IS NOT NULL AND provincia IS NOT NULL
GROUP BY provincia, departamento
ORDER BY casos_acumulados DESC
LIMIT 10
"""

VISTAS["vista_estadisticas"] = """
SELECT MIN(cant_casos) AS min_casos, MAX(cant_casos) AS max_casos,
       AVG(cant_casos) AS promedio_casos, COUNT(*) AS registros
FROM vig_dengue
"""

VISTAS["vista_incidencia_por_provincia"] = """
-- Incidencia acumulada del período 2018-2025: casos totales por 100.000 habitantes de la provincia
SELECT provincia,
       SUM(cant_casos) AS casos_acumulados,
       MAX(cant_habitantes) AS poblacion_provincial,
       SUM(cant_casos) * 100000.0 / NULLIF(MAX(cant_habitantes), 0) AS incidencia_acum
FROM vig_dengue
WHERE provincia IS NOT NULL
GROUP BY provincia
ORDER BY incidencia_acum DESC
"""

VISTAS["vista_incidencia_por_SE_provincia"] = """
-- Casos acumulados en cada semana epidemiológica (sumando todos los años) por 100.000 habitantes
SELECT SE, provincia,
       SUM(cant_casos) AS casos_acumulados,
       MAX(cant_habitantes) AS poblacion_provincial,
       SUM(cant_casos) * 100000.0 / NULLIF(MAX(cant_habitantes), 0) AS incidencia_acum
FROM vig_dengue
WHERE provincia IS NOT NULL
GROUP BY SE, provincia
ORDER BY provincia, CAST(SE AS INTEGER)
"""

VISTAS["vista_incidencia_por_grupo_edad"] = """
-- OJO: no es incidencia específica por edad, porque la fuente no trae población por grupo etario.
-- Relaciona los casos de cada grupo con la población total de las provincias con casos.
SELECT grupo_edad,
       SUM(cant_casos) AS casos_acumulados,
       (SELECT SUM(habitantes) FROM (SELECT MAX(cant_habitantes) AS habitantes FROM vig_dengue
                                     WHERE provincia IS NOT NULL GROUP BY provincia)) AS poblacion_total,
       SUM(cant_casos) * 100000.0 /
           (SELECT SUM(habitantes) FROM (SELECT MAX(cant_habitantes) AS habitantes FROM vig_dengue
                                         WHERE provincia IS NOT NULL GROUP BY provincia)) AS incidencia_acum
FROM vig_dengue
GROUP BY grupo_edad
"""

VISTAS["vista_variacion_anual"] = """
WITH casos_anuales AS (
    SELECT "año", SUM(cant_casos) AS casos_acumulados FROM vig_dengue GROUP BY "año"
)
SELECT "año", casos_acumulados,
       LAG(casos_acumulados) OVER (ORDER BY "año") AS casos_previos,
       casos_acumulados - LAG(casos_acumulados) OVER (ORDER BY "año") AS variacion
FROM casos_anuales
ORDER BY "año"
"""

# Las dos vistas siguientes salen del panel provincia-semana, que incluye las semanas con cero casos.
VISTAS["vista_variacion_semanal"] = """
SELECT "año", SE, provincia,
       cant_casos AS casos_acumulados,
       LAG(cant_casos) OVER (PARTITION BY provincia, "año" ORDER BY SE) AS casos_previos,
       cant_casos - LAG(cant_casos) OVER (PARTITION BY provincia, "año" ORDER BY SE) AS variacion
FROM panel_provincia_semana
ORDER BY provincia, "año", SE
"""

VISTAS["vista_clima_casos"] = """
-- Clima semanal real de la provincia (promedio de sus estaciones SMN) junto a los casos de la misma semana
SELECT provincia, "año", SE,
       temp_media AS temp_media_prom,
       precip_sem AS precip_media_prom,
       hum_rel AS hum_rel_prom,
       cant_casos AS casos_acumulados
FROM panel_provincia_semana
ORDER BY provincia, "año", SE
"""

VISTAS["vista_incidencia_por_var_clim"] = """
-- Una fila por provincia y semana (incluye semanas sin casos): es la tabla de base para relacionar clima e incidencia
SELECT SE, provincia, "año",
       temp_media AS temp_media_prom,
       precip_sem AS precip_media_prom,
       hum_rel AS hum_rel_prom,
       dias_lluvia_sem AS dias_pp_mas_1mm_prom,
       cant_casos AS casos_acumulados,
       cant_habitantes AS poblacion_provincial,
       incidencia_100k AS incidencia_acum
FROM panel_provincia_semana
"""

for nombre, sql in VISTAS.items():
    con.execute(f"CREATE VIEW {nombre} AS {sql.strip()}")
con.commit()

vistas_creadas = [f[0] for f in con.execute("SELECT name FROM sqlite_master WHERE type = 'view' ORDER BY name")]
print(f"Vistas creadas: {len(vistas_creadas)}")
for v in vistas_creadas:
    print(" -", v)

Vistas creadas: 13
 - vista_casos_por_anio
 - vista_casos_por_grupo_edad
 - vista_casos_por_provincia
 - vista_clima_casos
 - vista_estadisticas
 - vista_incidencia_por_SE_provincia
 - vista_incidencia_por_grupo_edad
 - vista_incidencia_por_provincia
 - vista_incidencia_por_var_clim
 - vista_proporcion_casos_grupo_edad
 - vista_top_departamentos
 - vista_variacion_anual
 - vista_variacion_semanal


Las consultas también se exportan a un archivo `.sql` legible, para poder consultarlas sin abrir el notebook.

In [10]:
# Las consultas también quedan en un archivo .sql legible, sin necesidad de abrir el notebook
partes = ["-- Vistas analíticas del proyecto de vigilancia del dengue (generado por 03_modelado_sql.ipynb)\n"]
for nombre, sql in VISTAS.items():
    partes.append(f"DROP VIEW IF EXISTS {nombre};\nCREATE VIEW {nombre} AS{sql.rstrip()};\n")
(ROOT / "sql" / "vistas_dengue.sql").write_text("\n".join(partes), encoding="utf-8")
print("Guardado: sql/vistas_dengue.sql")

Guardado: sql/vistas_dengue.sql


## 6. Verificación

### 6.1 Ejecución de todas las vistas

In [11]:
def vista(nombre):
    return pd.read_sql_query(f"SELECT * FROM {nombre}", con)

# Todas las vistas deben ejecutarse sin error y sin filas con provincia nula
for v in vistas_creadas:
    df_v = vista(v)
    nulos_prov = int(df_v["provincia"].isna().sum()) if "provincia" in df_v.columns else 0
    assert nulos_prov == 0, f"{v} tiene provincia nula"
    print(f"{v:<38} {len(df_v):>6} filas")

vista_casos_por_anio                        8 filas
vista_casos_por_grupo_edad                 13 filas
vista_casos_por_provincia                  23 filas
vista_clima_casos                        8234 filas
vista_estadisticas                          1 filas
vista_incidencia_por_SE_provincia         847 filas


vista_incidencia_por_grupo_edad            13 filas
vista_incidencia_por_provincia             23 filas
vista_incidencia_por_var_clim            8234 filas
vista_proporcion_casos_grupo_edad          13 filas
vista_top_departamentos                    10 filas
vista_variacion_anual                       8 filas
vista_variacion_semanal                  8234 filas


### 6.2 Comparación con pandas

Cada vista clave se recalcula en pandas de forma independiente y se compara con el resultado de SQL. Si las dos rutas coinciden, es muy poco probable que ambas tengan el mismo error.

In [12]:
# Cada vista se compara con el mismo cálculo hecho directamente en pandas
con_prov = fact[fact["provincia"].notna()]

# Casos por año
v = vista("vista_casos_por_anio").set_index("año")["casos_acumulados"]
p = fact.groupby(fact["año"].astype(str))["cant_casos"].sum()
assert (v.sort_index() == p.sort_index()).all()

# Casos e incidencia por provincia
v = vista("vista_incidencia_por_provincia").set_index("provincia")
p = con_prov.groupby("provincia").agg(casos=("cant_casos", "sum"), hab=("cant_habitantes", "max"))
p["inc"] = p["casos"] / p["hab"] * 100_000
assert (v["casos_acumulados"].sort_index() == p["casos"].sort_index()).all()
assert np.allclose(v["incidencia_acum"].sort_index(), p["inc"].sort_index())

# Proporción por grupo de edad: suma 100 %
assert abs(vista("vista_proporcion_casos_grupo_edad")["proporcion"].sum() - 100) < 1e-9

# Panel: casos y incidencia por provincia-semana
v = vista("vista_incidencia_por_var_clim")
assert v["casos_acumulados"].sum() == con_prov["cant_casos"].sum()
assert np.allclose(v["incidencia_acum"], v["casos_acumulados"] / v["poblacion_provincial"] * 100_000)

# Variación semanal: la primera semana de cada provincia y año no tiene semana previa
v = vista("vista_variacion_semanal")
assert v[v["SE"] == 1]["casos_previos"].isna().all()
print("Las vistas coinciden con el cálculo en pandas.")

Las vistas coinciden con el cálculo en pandas.


Las vistas coinciden con el cálculo independiente: casos por año, casos e incidencia por provincia, proporciones por grupo de edad, y el panel con sus incidencias y variaciones.

**Una limitación a tener presente.** La vista `vista_incidencia_por_grupo_edad` **no es una incidencia específica por edad**. Esa medida exige la población de cada grupo etario, y la fuente solo trae la población total de cada provincia. La vista divide los casos de cada grupo por la población total, lo que sirve para comparar el peso relativo de los grupos pero no su riesgo. Para un riesgo por edad haría falta cruzar los casos con una proyección de población por edad del INDEC.

## 7. Exportación para Power BI

Power BI no necesita conectarse a la base. Los datos de este proyecto son un corte histórico que no se modifica: no hay usuarios escribiendo, ni actualizaciones, ni consultas simultáneas, que son los motivos por los que se justifica un servidor de base de datos. Además SQLite no es un servidor sino un archivo, y Power BI no trae un conector nativo para leerlo: pide instalar un controlador ODBC en cada computadora que quiera refrescar el tablero, un obstáculo innecesario para quien clone el repositorio.

Por eso SQL cumple acá el rol que mejor le sale, el de **capa de modelado y consulta**: define las vistas, las valida y las documenta. Cada vista se exporta como un CSV con el nombre de la vista, en la carpeta `powerbi/`, y es lo que carga el dashboard. Se exporta también la tabla de hechos (`vig_dengue`), sobre la que se calculan las medidas.

In [13]:
POWERBI = ROOT / "powerbi"
POWERBI.mkdir(exist_ok=True)

def enteros_sin_decimal(df):
    # Una columna entera con algún nulo llega de SQL como decimal (1703186.0): se devuelve a entero para que el CSV quede limpio
    for col in df.select_dtypes("float").columns:
        valores = df[col].dropna()
        if len(valores) and (valores % 1 == 0).all():
            df[col] = df[col].astype("Int64")
    return df.round(4)

EXPORTAR = ["vig_dengue"] + vistas_creadas
filas_exportadas = {}
for nombre in EXPORTAR:
    df_x = enteros_sin_decimal(pd.read_sql_query(f'SELECT * FROM "{nombre}"', con))
    df_x.to_csv(POWERBI / f"{nombre}.csv", index=False, encoding="utf-8")
    filas_exportadas[nombre] = len(df_x)

print(f"Exportados {len(EXPORTAR)} archivos a powerbi/")
pd.Series(filas_exportadas, name="filas").to_frame()

Exportados 14 archivos a powerbi/


,filas
vig_dengue,70194
vista_casos_por_anio,8
vista_casos_por_grupo_edad,13
vista_casos_por_provincia,23
vista_clima_casos,8234
vista_estadisticas,1
vista_incidencia_por_SE_provincia,847
vista_incidencia_por_grupo_edad,13
vista_incidencia_por_provincia,23
vista_incidencia_por_var_clim,8234


In [14]:
# Cada CSV se vuelve a leer y se compara con su tabla o vista en la base: mismas filas, mismas columnas
for nombre in EXPORTAR:
    en_db = pd.read_sql_query(f'SELECT * FROM "{nombre}" LIMIT 1', con)
    en_csv = pd.read_csv(POWERBI / f"{nombre}.csv", nrows=1)
    n_db = con.execute(f'SELECT COUNT(*) FROM "{nombre}"').fetchone()[0]
    n_csv = sum(1 for _ in open(POWERBI / f"{nombre}.csv", encoding="utf-8")) - 1
    assert list(en_db.columns) == list(en_csv.columns), f"{nombre}: las columnas no coinciden"
    assert n_db == n_csv, f"{nombre}: {n_db} filas en la base y {n_csv} en el CSV"
print("Los CSV de powerbi/ coinciden con la base: mismas columnas y mismas filas.")

Los CSV de powerbi/ coinciden con la base: mismas columnas y mismas filas.


## 8. Resultado

In [15]:
resumen = vista("vista_incidencia_por_provincia")
resumen["casos_acumulados"] = resumen["casos_acumulados"].astype(int)
resumen["incidencia_acum"] = resumen["incidencia_acum"].round(0).astype(int)
resumen.head(8)

,provincia,casos_acumulados,poblacion_provincial,incidencia_acum
0,Tucumán,117185,1703186,6880
1,Chaco,43911,1142963,3842
2,La Rioja,14351,384607,3731
3,Formosa,21384,606041,3528
4,Salta,50347,1440672,3495
5,Santiago del Estero,34932,1054028,3314
6,Jujuy,25521,797955,3198
7,Córdoba,119520,3978984,3004


**Hallazgo:** Tucumán lidera la incidencia acumulada 2018–2025, con unos 6.880 casos por 100.000 habitantes. Pero esa cifra se concentra en pocas temporadas: el 71 % de los casos de Tucumán corresponde a 2024, y casi nada a los años sin epidemia. Es un acumulado de brotes, no una carga constante, algo que se analiza en el notebook 04.

In [16]:
con.close()
print("Base guardada en", DB.relative_to(ROOT), f"({DB.stat().st_size / 1e6:.1f} MB)")

Base guardada en sql/dengue.db (11.5 MB)


## Cierre

La base `sql/dengue.db` (11,5 MB) contiene la tabla de hechos con clima, el panel provincia-semana y las 13 vistas verificadas. Todo lo que lee Power BI está exportado como CSV en `powerbi/`, con el nombre de cada tabla o vista.

El siguiente notebook usa el panel para responder la pregunta central del proyecto: **¿qué relación hay entre el clima y los brotes de dengue?**